# 🎛️ Separador de stems (RoFormer + Demucs)

Separa canciones en stems listos para cortar y mandar a pads. Tú dices **qué quieres sacar** y el notebook elige el mejor modelo disponible para eso.

**Cómo se usa:** ejecuta las celdas **en orden** con el botón ▶️ de cada una. Las opciones se eligen en los formularios; no hay que tocar código.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar*. Sin GPU funciona solo con Demucs y mucho más lento.

Todo corre en Colab. Lo único externo es la descarga de los modelos (una vez por sesión) y, si tú lo eliges, tu Google Drive. Los archivos de la sesión se borran cuando Colab se desconecta: **descarga tus ZIP antes de cerrar**.

## 1 · Preparación
Revisa si hay GPU, instala los dos motores con versiones fijas (Demucs 4.1.0 y audio-separator 0.48.0, ambos MIT) y carga las funciones. **No hay nada que tocar.** La primera vez tarda unos minutos; si vuelves a ejecutarla no reinstala nada.

In [ ]:
#@title 1 · Preparación: revisar GPU, instalar Demucs y cargar funciones { display-mode: "form" }
import gc, importlib, importlib.metadata, importlib.util, json, math, os, re, shutil
import subprocess, sys, time, warnings, zipfile
from pathlib import Path

VERSION_DEMUCS = "4.1.0"            # ya no depende de torchaudio para leer/guardar audio
VERSION_AUDIO_SEPARATOR = "0.48.0"  # motor para los modelos RoFormer (BS-Roformer, MelBand Roformer)

import torch   # Colab ya lo trae; NO se reinstala
_torch_antes = importlib.metadata.version("torch")


def _version(paquete):
    try:
        return importlib.metadata.version(paquete)
    except importlib.metadata.PackageNotFoundError:
        return None


def _falta(modulos):
    return [m for m in modulos if importlib.util.find_spec(m) is None]


def _pip(paquetes, obligatorio):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paquetes], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1000:], r.stderr[-3000:])
        if obligatorio:
            raise RuntimeError("❌ No se pudo instalar. Revisa tu conexión y vuelve a ejecutar esta celda.")
    importlib.invalidate_caches()
    return r.returncode == 0


if _version("demucs") == VERSION_DEMUCS and not _falta(["soundfile", "julius", "sphn", "lameenc"]):
    print(f"✅ Demucs {VERSION_DEMUCS} ya estaba instalado.")
else:
    print(f"⏳ Instalando Demucs {VERSION_DEMUCS}…")
    _pip([f"demucs=={VERSION_DEMUCS}", "soundfile"], obligatorio=True)
    print(f"✅ Demucs {VERSION_DEMUCS} instalado.")

if _version("audio-separator") == VERSION_AUDIO_SEPARATOR and not _falta(["audioread", "onnxruntime"]):
    print(f"✅ audio-separator {VERSION_AUDIO_SEPARATOR} ya estaba instalado.")
else:
    print(f"⏳ Instalando audio-separator {VERSION_AUDIO_SEPARATOR} para los modelos RoFormer (2-3 minutos, solo la primera vez)…")
    # audioread: lo necesita audio-separator y las versiones nuevas de librosa ya no lo traen.
    # onnxruntime (versión CPU, ligera): solo se agrega si no hay ninguno; los RoFormer corren en PyTorch con la GPU.
    _extra = [] if importlib.util.find_spec("onnxruntime") else ["onnxruntime"]
    if _pip([f"audio-separator=={VERSION_AUDIO_SEPARATOR}", "audioread", *_extra], obligatorio=False):
        print(f"✅ audio-separator {VERSION_AUDIO_SEPARATOR} instalado.")

if _version("torch") != _torch_antes:
    print(f"⚠️ La instalación cambió PyTorch ({_torch_antes} → {_version('torch')}). Ve a «Entorno de ejecución → "
          "Reiniciar sesión» y ejecuta esta celda otra vez.")

warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

import numpy as np
import soundfile as sf
import julius
import torch.nn.functional as F
from tqdm.auto import tqdm
from demucs.pretrained import get_model
from demucs.apply import apply_model
from demucs.audio import convert_audio
from demucs.htdemucs import HTDemucs

import logging
try:
    from audio_separator.separator import Separator as SeparadorRoformer
    HAY_ROFORMER = True
except Exception as _error:
    SeparadorRoformer = None
    HAY_ROFORMER = False
    print(f"⚠️ audio-separator no está disponible ({type(_error).__name__}: {str(_error)[:150]}).\n"
          "   El notebook sigue funcionando, pero solo con los modelos de Demucs.")

try:
    from google.colab import files as colab_files, drive as colab_drive
    EN_COLAB = True
except ImportError:
    colab_files = colab_drive = None
    EN_COLAB = False

# ---------- Carpetas de trabajo (todo es temporal: vive en la sesión de Colab) ----------
BASE = Path(os.environ.get("STEMS_BASE", "/content/stems_demucs" if EN_COLAB else "stems_demucs")).resolve()
DIR_ENTRADA = BASE / "entrada"     # archivos subidos
DIR_SALIDA = BASE / "salida"       # una carpeta por canción con sus stems
DIR_ZIPS = BASE / "zips"           # ZIPs para descargar
DIR_PREVIEWS = BASE / "previews"   # MP3 ligeros para la celda de escuchar
DIR_MODELOS = BASE / "modelos"     # pesos de los modelos RoFormer (se descargan una vez por sesión)
DIR_TMP = BASE / "tmp"             # archivos de paso entre modelos
for _d in (DIR_ENTRADA, DIR_SALIDA, DIR_ZIPS, DIR_PREVIEWS, DIR_MODELOS, DIR_TMP):
    _d.mkdir(parents=True, exist_ok=True)

EXTENSIONES = (".wav", ".mp3", ".flac", ".m4a", ".ogg")
ORDEN_STEMS = ["vocals", "drums", "bass", "guitar", "piano", "other", "instrumental"]
SR_TRABAJO = 44100   # todos los modelos trabajan a 44.1 kHz; la salida vuelve a la frecuencia original

# Canciones largas: se procesan por bloques para no llenar la memoria RAM.
UN_BLOQUE_MAX_S = 600   # hasta 10 min se procesa de una sola pasada
BLOQUE_S = 300          # si es más larga: bloques de 5 min…
SOLAPE_S = 10           # …que se traslapan 10 s y se funden entre sí


class ErrorDeAudio(Exception):
    """Problema con un archivo de audio (mensaje ya entendible)."""


# ---------- Utilidades ----------
def fmt_tiempo(segundos):
    segundos = int(round(segundos))
    return f"{segundos // 60}:{segundos % 60:02d}"


def nombre_limpio(nombre):
    base = re.sub(r'[\\/:*?"<>|\x00-\x1f]', "_", str(nombre))
    base = re.sub(r"\s+", " ", base).strip(" .")
    return base[:120].strip(" .") or "cancion"


def nombres_unicos(rutas):
    """Un nombre de canción distinto para cada archivo (aunque dos se llamen igual)."""
    usados, resultado = set(), {}
    for ruta in rutas:
        ruta = Path(ruta)
        nombre = nombre_limpio(ruta.stem)
        if nombre.lower() in usados:
            nombre = f"{nombre} ({ruta.suffix.lstrip('.').lower()})"
        base, n = nombre, 2
        while nombre.lower() in usados:
            nombre = f"{base} {n}"
            n += 1
        usados.add(nombre.lower())
        resultado[str(ruta)] = nombre
    return resultado


def listar_audios(carpeta):
    carpeta = Path(carpeta)
    todos = sorted([p for p in carpeta.iterdir() if p.is_file() and not p.name.startswith(".")],
                   key=lambda p: p.name.lower())
    buenos = [p for p in todos if p.suffix.lower() in EXTENSIONES]
    ignorados = [p for p in todos if p.suffix.lower() not in EXTENSIONES]
    return buenos, ignorados


def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def es_falta_de_memoria_gpu(error):
    oom = getattr(torch, "OutOfMemoryError", None) or getattr(torch.cuda, "OutOfMemoryError", ())
    return isinstance(error, oom) or "cuda out of memory" in str(error).lower()


def explicar_error(error):
    """Convierte un error técnico en un mensaje entendible."""
    texto = str(error)
    if isinstance(error, ErrorDeAudio):
        return texto
    if isinstance(error, SystemExit):
        return "no se pudo obtener el modelo (¿falló la descarga?)."
    if es_falta_de_memoria_gpu(error):
        return ("se acabó la memoria de la GPU. Usa «Entorno de ejecución → Reiniciar sesión» para "
                "liberarla y vuelve a intentar con esa canción sola.")
    if isinstance(error, MemoryError) or "allocate memory" in texto or "not enough memory" in texto.lower():
        return ("se acabó la memoria RAM de Colab. La canción es demasiado larga o pesada: "
                "córtala en partes o reinicia la sesión y procésala sola.")
    if isinstance(error, FileNotFoundError):
        return f"no encuentro el archivo ({texto}). ¿Lo borraste o moviste? Vuelve a ejecutar la celda 2."
    if isinstance(error, OSError) and getattr(error, "errno", None) == 28:
        return "se llenó el disco de Colab. Ejecuta la celda de limpieza y vuelve a intentar."
    return f"{type(error).__name__}: {texto[:300]}"


# ---------- Lectura de audio (con ffmpeg, sin torchaudio) ----------
def info_audio(ruta):
    r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "a:0", "-show_entries",
                        "stream=sample_rate,channels", "-of", "json", str(ruta)],
                       capture_output=True, text=True)
    try:
        pista = json.loads(r.stdout or "{}").get("streams", [])[0]
        return int(pista["sample_rate"]), int(pista["channels"])
    except Exception:
        raise ErrorDeAudio("no se pudo leer: el archivo está dañado, no es audio o su formato no es compatible.")


def leer_audio(ruta):
    """Devuelve (audio estéreo float32 [2, muestras], frecuencia de muestreo original)."""
    if not Path(ruta).is_file():
        raise FileNotFoundError(str(ruta))
    sr, canales = info_audio(ruta)
    cmd = ["ffmpeg", "-nostdin", "-v", "error", "-i", str(ruta), "-map", "0:a:0",
           "-f", "f32le", "-acodec", "pcm_f32le"]
    if canales > 2:            # 5.1 y similares: ffmpeg hace la mezcla a estéreo
        cmd += ["-ac", "2"]
        canales = 2
    r = subprocess.run(cmd + ["-"], capture_output=True)
    if r.returncode != 0 or len(r.stdout) < 4 * canales:
        detalle = r.stderr.decode("utf-8", "ignore").strip().splitlines()[-1:] or [""]
        raise ErrorDeAudio(f"ffmpeg no pudo decodificar el archivo. {detalle[0][:200]}")
    datos = np.frombuffer(r.stdout, dtype="<f4")
    del r
    datos = datos[: len(datos) // canales * canales].reshape(-1, canales).T
    wav = torch.from_numpy(np.ascontiguousarray(datos, dtype=np.float32))
    del datos
    if canales == 1:
        wav = wav.repeat(2, 1)   # mono → mismo audio en ambos canales
    if wav.shape[1] < sr // 2:
        raise ErrorDeAudio("el audio dura menos de medio segundo.")
    if not torch.isfinite(wav).all():
        wav = torch.nan_to_num(wav, nan=0.0, posinf=1.0, neginf=-1.0)
    return wav, sr


# ---------- Catálogo de modelos (cifras: SDR en el conjunto Multisong de MVSEP; más alto = mejor) ----------
FUENTE_MVSEP = "https://mvsep.com/en/algorithms"
MODELOS = {
    "sw": {"nombre": "BS-Roformer-SW (6 stems)", "archivo": "BS-Roformer-SW.ckpt",
           "licencia": "DESCONOCIDA: pesos compartidos sin autor ni licencia declarados (el código sí es MIT)"},
    "kim": {"nombre": "MelBand Roformer de Kimberley Jensen (voz)", "archivo": "vocals_mel_band_roformer.ckpt",
            "licencia": "MIT según su repositorio en Hugging Face (KimberleyJSN/melbandroformer)"},
    "htdemucs_ft": {"nombre": "Demucs htdemucs_ft", "licencia": "MIT (Meta); los pesos no traen una licencia aparte"},
    "htdemucs": {"nombre": "Demucs htdemucs", "licencia": "MIT (Meta); los pesos no traen una licencia aparte"},
    "htdemucs_6s": {"nombre": "Demucs htdemucs_6s", "licencia": "MIT (Meta); los pesos no traen una licencia aparte"},
}
OBJETIVOS = {
    "voz": ["vocals", "instrumental"],
    "4": ["drums", "bass", "vocals", "other"],
    "6": ["drums", "bass", "vocals", "guitar", "piano", "other"],
    "drums": ["drums"],
}
FAMILIAS = {"sw": "BS-Roformer-SW", "kim": "MelBand Roformer Kim (voz) + Demucs (resto)",
            "demucs": "Solo Demucs (htdemucs_ft / htdemucs_6s)", "demucs_rapido": "Solo Demucs rápido (htdemucs)"}


def _demucs_de(familia, objetivo):
    if objetivo == "6":
        return "htdemucs_6s"
    return "htdemucs" if familia == "demucs_rapido" else "htdemucs_ft"


def elegir_plan(objetivo, forzar=None, solo_licencias_claras=False, hay_gpu=True, hay_roformer=True):
    """Decide qué modelo(s) usar para un objetivo. Devuelve un dict con la familia, los modelos y el porqué."""
    avisos = []
    if forzar:
        familia, motivo = forzar, "lo forzaste en «Avanzado»."
        if familia in ("sw", "kim") and not hay_roformer:
            familia = "demucs"
            avisos.append("audio-separator no está instalado: se usa Demucs en lugar del modelo forzado.")
        elif familia in ("sw", "kim") and not hay_gpu:
            avisos.append("los modelos RoFormer en CPU son lentísimos (unas 4 veces la duración de la canción o más).")
    elif not hay_roformer or not hay_gpu:
        familia = "demucs_rapido" if (objetivo == "4" and not hay_gpu) else "demucs"
        motivo = ("sin GPU los modelos RoFormer tardarían demasiado, así que se usa Demucs." if hay_roformer
                  else "audio-separator no está disponible, así que se usa Demucs.")
        avisos.append("con GPU se elegiría un modelo RoFormer, de mejor calidad.")
    elif objetivo == "voz":
        familia = "kim"
        motivo = ("en MVSEP los RoFormer superan a Demucs por ~2.7 dB en voz (MelBand Kim 11.01 vs htdemucs_ft 8.33). "
                  "BS-Roformer-SW marca 11.30, pero 0.3 dB no compensa usar pesos de licencia desconocida.")
    elif solo_licencias_claras:
        familia = "demucs" if objetivo == "drums" else "kim"
        motivo = ("pediste solo licencias claras: " + (
            "batería con el sub-modelo de batería de htdemucs_ft (11.24 dB en MVSEP)." if objetivo == "drums" else
            "la voz la saca MelBand Kim (11.01 dB) y Demucs separa el resto sobre el instrumental ya sin voz."))
    else:
        familia = "sw"
        motivo = {"4": "en MVSEP supera a htdemucs_ft en todo: batería 14.11 vs 11.24, bajo 14.62 vs 12.05, otros 8.71 vs 5.74, voz 11.30 vs 8.33.",
                  "6": "da guitarra (9.05 dB) y piano (7.83 dB) y supera a htdemucs_6s (batería 14.11 vs 10.59, bajo 14.62 vs 11.42); Demucs admite que su piano sale con artefactos.",
                  "drums": "es el mejor modelo público de batería en MVSEP: 14.11 dB vs 11.24 de htdemucs_ft."}[objetivo]

    if familia == "sw":
        modelos = ["sw"]
    elif familia == "kim":
        modelos = ["kim"] if objetivo == "voz" else ["kim", _demucs_de(familia, objetivo)]
    else:
        modelos = [_demucs_de(familia, objetivo)]
    if "sw" in modelos:
        avisos.append("BS-Roformer-SW no tiene licencia conocida. Para uso personal funciona, pero no es «libre»: "
                      "marca «solo_licencias_claras» si prefieres evitarlo.")
    if objetivo == "6" and "htdemucs_6s" in modelos:
        avisos.append("el piano de htdemucs_6s suele salir con artefactos (lo dice el propio proyecto Demucs).")
    return {"familia": familia, "objetivo": objetivo, "salidas": list(OBJETIVOS[objetivo]), "modelos": modelos,
            "motivo": motivo, "avisos": avisos, "usa_roformer": any(m in ("sw", "kim") for m in modelos)}


def describir_plan(plan):
    lineas = ["🧠 Modelo(s): " + "  +  ".join(MODELOS[m]["nombre"] for m in plan["modelos"]),
              "   Por qué: " + plan["motivo"], f"   Fuente de las cifras: {FUENTE_MVSEP} (conjunto Multisong)"]
    lineas += [f"   Licencia de {MODELOS[m]['nombre']}: {MODELOS[m]['licencia']}" for m in plan["modelos"]]
    lineas += [f"   ⚠️ {a}" for a in plan["avisos"]]
    return "\n".join(lineas)


# ---------- Carga de modelos (se conservan si vuelves a ejecutar la celda) ----------
_MODELOS = globals().get("_MODELOS", {})          # Demucs
_ROFORMER = globals().get("_ROFORMER", {})        # audio-separator: un modelo a la vez


def descargar_modelos_de_memoria():
    _MODELOS.clear()
    _ROFORMER.clear()
    liberar_memoria()


def cargar_demucs(nombre):
    if nombre not in _MODELOS:
        _MODELOS.clear()        # solo un modelo de Demucs en memoria a la vez
        liberar_memoria()
        print(f"   ⬇️ Cargando {MODELOS.get(nombre, {}).get('nombre', nombre)} (la primera vez se descarga)…")
        modelo = get_model(nombre)
        modelo.eval()
        _MODELOS[nombre] = modelo
    return _MODELOS[nombre]


def cargar_roformer(clave, overlap=0):
    """Devuelve un separador de audio-separator con el modelo RoFormer ya cargado."""
    if not HAY_ROFORMER:
        raise RuntimeError("audio-separator no está instalado")
    firma = (clave, int(overlap))
    if _ROFORMER.get("firma") != firma:
        _ROFORMER.clear()
        liberar_memoria()
        print(f"   ⬇️ Cargando {MODELOS[clave]['nombre']} (la primera vez descarga ~0.7-0.9 GB)…")
        salida = DIR_TMP / "roformer_out"
        salida.mkdir(parents=True, exist_ok=True)
        sep = SeparadorRoformer(
            log_level=logging.ERROR, model_file_dir=str(DIR_MODELOS), output_dir=str(salida),
            output_format="WAV", use_soundfile=True, normalization_threshold=1.0, sample_rate=SR_TRABAJO,
            mdxc_params={"segment_size": 256, "override_model_segment_size": False, "batch_size": 1,
                         "overlap": int(overlap) if overlap else None, "pitch_shift": 0})
        sep.load_model(model_filename=MODELOS[clave]["archivo"])
        # audio-separator baja el volumen de un stem si pasa de 1.0; aquí se desactiva para que
        # los stems conserven su nivel real (el control de picos se hace al final, parejo para todos).
        if getattr(sep, "model_instance", None) is not None:
            sep.model_instance.normalization_threshold = float("inf")
        _ROFORMER.update(firma=firma, sep=sep, salida=salida)
    return _ROFORMER["sep"], _ROFORMER["salida"]


def segmento_maximo(modelo):
    if hasattr(modelo, "max_allowed_segment"):
        return float(modelo.max_allowed_segment)
    if isinstance(modelo, HTDemucs):
        return float(modelo.segment)
    return float("inf")


# ---------- Motores: reciben la mezcla a 44.1 kHz [2, muestras] y devuelven {stem: audio} ----------
def motor_demucs(nombre, mezcla, cfg, dispositivo, solo=None):
    """Demucs. Con `solo`, y si el modelo es un conjunto con un sub-modelo por stem (htdemucs_ft),
    se ejecuta únicamente el sub-modelo de ese stem (4 veces más rápido, mismo resultado)."""
    modelo = cargar_demucs(nombre)
    fuentes = list(modelo.sources)
    red, indice = modelo, None
    if solo is not None:
        indice = fuentes.index(solo)
        if hasattr(modelo, "models"):
            utiles = [i for i, pesos in enumerate(modelo.weights) if pesos[indice] > 0]
            if len(utiles) == 1:
                red = modelo.models[utiles[0]]
    maximo = segmento_maximo(red)
    segmento = max(1.0, min(float(cfg["segment"]), maximo))
    shifts, overlap = int(cfg["shifts"]), float(cfg["overlap"])
    n_modelos = len(red.models) if hasattr(red, "models") else 1
    paso = max(1, int((1 - overlap) * int(SR_TRABAJO * segmento)))
    extra = int(0.5 * SR_TRABAJO) if shifts else 0
    total = n_modelos * max(1, shifts) * math.ceil((mezcla.shape[-1] + extra) / paso)

    ref = mezcla.mean(0)                      # misma normalización que usa Demucs (se deshace al final)
    media, desv = float(ref.mean()), float(ref.std()) + 1e-8
    del ref
    with tqdm(total=total, desc=f"   {nombre}", unit="trozo", leave=False) as barra:
        def avance(datos):
            if datos.get("state") == "end" and barra.n < barra.total:
                barra.update(1)
        est = apply_model(red, ((mezcla - media) / desv)[None], shifts=shifts, split=True, overlap=overlap,
                          segment=None if segmento >= maximo else segmento, device=dispositivo,
                          progress=False, callback=avance)[0]
    est.mul_(desv).add_(media)
    if solo is not None:
        return {solo: est[indice].clone()}
    return {f: est[i] for i, f in enumerate(fuentes)}


def motor_roformer(clave, mezcla, cfg):
    """RoFormer vía audio-separator: escribe la mezcla en un WAV float temporal y lee los stems que devuelve."""
    sep, salida = cargar_roformer(clave, cfg.get("overlap_roformer", 0))
    entrada = DIR_TMP / "roformer_in"
    for carpeta in (entrada, salida):
        shutil.rmtree(carpeta, ignore_errors=True)
        carpeta.mkdir(parents=True, exist_ok=True)
    largo = mezcla.shape[-1]
    try:
        sf.write(str(entrada / "bloque.wav"), mezcla.T.contiguous().numpy(), SR_TRABAJO, subtype="FLOAT")
        stems = {}
        for ruta in sep.separate(str(entrada / "bloque.wav")):
            ruta = Path(ruta) if os.path.isabs(str(ruta)) else salida / ruta
            encontrado = re.search(r"_\(([^()]+)\)_", ruta.name)
            if not encontrado:
                continue
            datos, sr = sf.read(str(ruta), dtype="float32", always_2d=True)
            if sr != SR_TRABAJO:
                raise RuntimeError(f"el modelo devolvió audio a {sr} Hz en vez de {SR_TRABAJO} Hz")
            audio = torch.from_numpy(np.ascontiguousarray(datos.T))
            if audio.shape[0] == 1:
                audio = audio.repeat(2, 1)
            if audio.shape[-1] > largo:
                audio = audio[..., :largo]
            elif audio.shape[-1] < largo:
                audio = F.pad(audio, (0, largo - audio.shape[-1]))
            stems[encontrado.group(1).strip().lower()] = audio
    finally:
        shutil.rmtree(entrada, ignore_errors=True)
        shutil.rmtree(salida, ignore_errors=True)
    if not stems:
        raise RuntimeError("el modelo RoFormer no devolvió ningún stem")
    return stems


def _pedir(stems, nombres, modelo):
    faltan = [n for n in nombres if n not in stems]
    if faltan:
        raise RuntimeError(f"{modelo} no devolvió: {', '.join(faltan)} (devolvió: {', '.join(stems)})")
    return [stems[n] for n in nombres]


def ejecutar_plan(plan, mezcla, cfg, dispositivo):
    """Aplica el plan a un bloque de audio (44.1 kHz) y devuelve {stem: audio} en el orden de salida."""
    familia, objetivo = plan["familia"], plan["objetivo"]
    if familia == "sw":
        s = motor_roformer("sw", mezcla, cfg)
        if objetivo == "voz":
            (voz,) = _pedir(s, ["vocals"], "BS-Roformer-SW")
            r = {"vocals": voz, "instrumental": mezcla - voz}      # instrumental = mezcla menos la voz
        elif objetivo == "drums":
            r = {"drums": _pedir(s, ["drums"], "BS-Roformer-SW")[0]}
        else:
            _pedir(s, OBJETIVOS["6"], "BS-Roformer-SW")
            if objetivo == "4":                                    # guitarra y piano se suman a «otros»
                s["other"] = s["other"] + s.pop("guitar") + s.pop("piano")
            r = s
    elif familia == "kim":
        (voz,) = _pedir(motor_roformer("kim", mezcla, cfg), ["vocals"], "MelBand Roformer Kim")
        instrumental = mezcla - voz
        if objetivo == "voz":
            r = {"vocals": voz, "instrumental": instrumental}
        elif objetivo == "drums":
            r = motor_demucs("htdemucs_ft", instrumental, cfg, dispositivo, solo="drums")
        else:                                                       # Demucs separa el instrumental ya sin voz
            r = motor_demucs(_demucs_de(familia, objetivo), instrumental, cfg, dispositivo)
            r["other"] = r["other"] + r.pop("vocals")               # restos de voz que encuentre → «otros»
            r["vocals"] = voz
    else:
        nombre = _demucs_de(familia, objetivo)
        if objetivo == "voz":
            voz = motor_demucs(nombre, mezcla, cfg, dispositivo, solo="vocals")["vocals"]
            r = {"vocals": voz, "instrumental": mezcla - voz}
        elif objetivo == "drums":
            r = motor_demucs(nombre, mezcla, cfg, dispositivo, solo="drums")
        else:
            r = motor_demucs(nombre, mezcla, cfg, dispositivo)
    return {nombre: r[nombre] for nombre in plan["salidas"]}


def plan_bloques(total, sr):
    """Divide canciones largas en bloques traslapados. Devuelve (bloques, traslape en muestras)."""
    if total <= int(UN_BLOQUE_MAX_S * sr):
        return [(0, total)], 0
    bloque, solape = int(BLOQUE_S * sr), int(SOLAPE_S * sr)
    return [(i, min(i + bloque, total)) for i in range(0, total - solape, bloque - solape)], solape


def nombre_stem(cancion, stem):
    return f"{cancion} - {stem}.wav"


def stems_de(carpeta, cancion):
    """Lista ordenada de los WAV finales de una canción."""
    carpeta = Path(carpeta)
    prefijo = f"{cancion} - "
    encontrados = [p for p in carpeta.glob("*.wav") if p.name.startswith(prefijo) and not p.name.startswith(".")]

    def clave(p):
        stem = p.stem[len(prefijo):]
        return (ORDEN_STEMS.index(stem) if stem in ORDEN_STEMS else len(ORDEN_STEMS), stem)
    return sorted(encontrados, key=clave)


def _terminar_wav(temporal, final, formato, ganancia):
    """Pasa el WAV temporal (float) al formato final. Solo al terminar aparece el archivo definitivo."""
    temporal, final = Path(temporal), Path(final)
    if formato == "float32" and ganancia == 1.0:
        os.replace(temporal, final)
        return
    subtipo = "FLOAT" if formato == "float32" else "PCM_24"
    parcial = final.with_name("." + final.name + ".part")
    with sf.SoundFile(str(temporal)) as origen, \
            sf.SoundFile(str(parcial), "w", samplerate=origen.samplerate, channels=origen.channels,
                         subtype=subtipo, format="WAV") as destino:
        for trozo in origen.blocks(blocksize=1 << 18, dtype="float32", always_2d=True):
            destino.write(trozo * ganancia if ganancia != 1.0 else trozo)
    os.replace(parcial, final)
    temporal.unlink()


def separar_cancion(ruta, cancion, carpeta, plan, cfg, dispositivo):
    """Separa UNA canción según `plan` y escribe sus stems en `carpeta`. Devuelve datos del resultado.

    cfg: dict con formato ("pcm24" | "float32"), segment, shifts, overlap, overlap_roformer.
    """
    carpeta = Path(carpeta)
    wav, sr = leer_audio(ruta)
    total = wav.shape[1]
    salidas = list(plan["salidas"])
    bloques, solape = plan_bloques(total, sr)

    carpeta.mkdir(parents=True, exist_ok=True)
    for viejo in list(carpeta.glob("*.wav")) + list(carpeta.glob(".*")) + [carpeta / "info.json"]:   # restos anteriores
        if viejo.is_file():
            viejo.unlink()
    temporales = {s: carpeta / f".tmp - {s}.wav" for s in salidas}
    escritores = {}
    pico, cola = 0.0, None
    try:
        for s in salidas:
            escritores[s] = sf.SoundFile(str(temporales[s]), "w", samplerate=sr, channels=2,
                                         subtype="FLOAT", format="WAV")
        for i, (a, b) in enumerate(bloques):
            if len(bloques) > 1:
                print(f"   · bloque {i + 1} de {len(bloques)}")
            mezcla = wav[:, a:b]
            if sr != SR_TRABAJO:
                mezcla = convert_audio(mezcla, sr, SR_TRABAJO, 2)
            stems = ejecutar_plan(plan, mezcla, cfg, dispositivo)
            del mezcla
            est = torch.stack([stems[s] for s in salidas])
            del stems
            if sr != SR_TRABAJO:
                est = julius.resample_frac(est, SR_TRABAJO, sr)      # de vuelta a la frecuencia original
            largo = b - a
            if est.shape[-1] > largo:
                est = est[..., :largo]
            elif est.shape[-1] < largo:
                est = F.pad(est, (0, largo - est.shape[-1]))
            if cola is not None:                                   # fundido con el bloque anterior
                rampa = torch.linspace(0.0, 1.0, solape)
                est[..., :solape] = est[..., :solape] * rampa + cola * (1.0 - rampa)
            if i < len(bloques) - 1:
                cola = est[..., -solape:].clone()
                listo = est[..., :-solape]
            else:
                listo = est
            pico = max(pico, float(listo.abs().max()))
            for k, s in enumerate(salidas):
                escritores[s].write(listo[k].T.contiguous().numpy())
            del est, listo
            liberar_memoria()
        for e in escritores.values():
            e.close()
        escritores = {}

        # Si algún stem pasa de 0 dBFS, en 24 bits se baja el volumen de TODOS los stems por igual
        # (así no se distorsiona y los stems conservan su balance entre sí).
        ganancia = 1.0
        if cfg["formato"] != "float32" and pico > 1.0:
            ganancia = 1.0 / (pico * 1.001)
        archivos = []
        for s in salidas:
            final = carpeta / nombre_stem(cancion, s)
            _terminar_wav(temporales[s], final, cfg["formato"], ganancia)
            archivos.append(final.name)
    finally:
        for e in escritores.values():
            try:
                e.close()
            except Exception:
                pass
        for t in temporales.values():
            if t.exists():
                t.unlink()
        for parcial in carpeta.glob(".*.part"):
            parcial.unlink()
        del wav
        liberar_memoria()

    return {"archivos": archivos, "sr": sr, "duracion_s": total / sr, "pico": pico,
            "ganancia_db": 20 * math.log10(ganancia), "bloques": len(bloques), "dispositivo": dispositivo}


# ---------- ZIP ----------
def crear_zip(destino, elementos):
    """elementos: lista de (ruta, nombre dentro del zip). No lo rehace si ya está al día."""
    destino = Path(destino)
    elementos = [(Path(r), n) for r, n in elementos]
    if destino.exists():
        try:
            with zipfile.ZipFile(destino) as z:
                mismos = sorted(z.namelist()) == sorted(n for _, n in elementos)
            if mismos and destino.stat().st_mtime >= max(r.stat().st_mtime for r, _ in elementos):
                return destino, False
        except Exception:
            pass
    destino.parent.mkdir(parents=True, exist_ok=True)
    parcial = destino.with_name("." + destino.name + ".part")
    try:
        with zipfile.ZipFile(parcial, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
            for ruta, nombre in elementos:
                z.write(ruta, nombre)
        os.replace(parcial, destino)
    finally:
        if parcial.exists():
            parcial.unlink()
    return destino, True


def copiar_si_cambio(origen, destino):
    origen, destino = Path(origen), Path(destino)
    if destino.exists() and destino.stat().st_size == origen.stat().st_size \
            and destino.stat().st_mtime >= origen.stat().st_mtime:
        return False
    destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(origen, destino)
    return True


def canciones_hechas():
    """Canciones que ya tienen stems en disco: lista de (nombre, carpeta, info)."""
    hechas = []
    for carpeta in sorted(DIR_SALIDA.iterdir(), key=lambda p: p.name.lower()):
        ficha = carpeta / "info.json"
        if carpeta.is_dir() and ficha.exists():
            try:
                info = json.loads(ficha.read_text(encoding="utf-8"))
            except Exception:
                continue
            if stems_de(carpeta, carpeta.name):
                hechas.append((carpeta.name, carpeta, info))
    return hechas


# ---------- Revisión del entorno ----------
print()
for _prog in ("ffmpeg", "ffprobe"):
    if shutil.which(_prog) is None:
        print(f"❌ Falta «{_prog}» (Colab lo trae de fábrica). Sin él no se pueden leer las canciones.")
print(f"PyTorch {torch.__version__} · Demucs {_version('demucs')} · audio-separator {_version('audio-separator') if HAY_ROFORMER else 'NO disponible'}")
if torch.cuda.is_available():
    _gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU detectada: {torch.cuda.get_device_name(0)} ({_gb:.0f} GB). Todo listo, sigue con la celda 2.")
else:
    print("⚠️ NO hay GPU en esta sesión: va a funcionar en CPU, pero MUCHO más lento y solo con Demucs "
          "(los modelos RoFormer, que son los mejores, necesitan GPU para ser prácticos).")
    print("   Para activarla: menú «Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4», "
          "guarda y vuelve a ejecutar esta celda.")

## 2 · Entrada
Elige **subir archivos** (wav, mp3, flac, m4a, ogg) o usar una **carpeta de Google Drive** (solo en ese caso se conecta Drive y te pide permiso). Puedes ejecutarla varias veces para agregar más canciones.

In [ ]:
#@title 2 · Entrada: elige de dónde salen las canciones { display-mode: "form" }
fuente = "Subir archivos desde mi compu"  #@param ["Subir archivos desde mi compu", "Carpeta de Google Drive"]
#@markdown **Si subes archivos:** desmarca esta casilla para NO pedir archivos otra vez y usar los que ya subiste.
pedir_archivos = True  #@param {type:"boolean"}
#@markdown **Si usas Drive:** carpeta donde están las canciones y carpeta donde se guardarán los stems (rutas dentro de tu Drive).
carpeta_drive_entrada = "MyDrive/stems_demucs/entrada"  #@param {type:"string"}
carpeta_drive_salida = "MyDrive/stems_demucs/salida"  #@param {type:"string"}


def _ruta_drive(texto):
    texto = str(texto).strip().replace("\\", "/").strip("/")
    for prefijo in ("content/drive/", "drive/"):
        if texto.startswith(prefijo):
            texto = texto[len(prefijo):]
    if not (texto.startswith("MyDrive") or texto.startswith("Shareddrives")):
        texto = "MyDrive/" + texto
    return Path("/content/drive") / texto


def _entrada():
    global ENTRADAS, DRIVE_SALIDA
    if "separar_cancion" not in globals():
        print("❌ Primero ejecuta la celda 1 (Preparación).")
        return
    ENTRADAS, DRIVE_SALIDA = [], None

    if fuente.startswith("Carpeta"):
        if not EN_COLAB:
            print("❌ Google Drive solo se puede montar dentro de Colab.")
            return
        if not os.path.ismount("/content/drive"):
            print("Conectando con Google Drive (Colab te pedirá permiso)…")
            try:
                colab_drive.mount("/content/drive")
            except Exception as error:
                print(f"❌ No se pudo conectar con Drive: {error}")
                return
        carpeta = _ruta_drive(carpeta_drive_entrada)
        if not carpeta.is_dir():
            print(f"❌ No existe la carpeta «{carpeta}». Créala en tu Drive, pon ahí las canciones "
                  "y vuelve a ejecutar esta celda (o corrige la ruta).")
            return
        DRIVE_SALIDA = _ruta_drive(carpeta_drive_salida)
        print(f"📂 Entrada: {carpeta}\n📂 Los stems se copiarán también a: {DRIVE_SALIDA}")
    else:
        carpeta = DIR_ENTRADA
        if pedir_archivos and EN_COLAB:
            print("Elige uno o varios archivos (wav, mp3, flac, m4a, ogg). Espera a que la subida llegue al 100 %.")
            try:
                subidos = colab_files.upload()
            except Exception as error:
                print(f"⚠️ La subida se interrumpió ({error}). Se usan los archivos que ya estaban.")
                subidos = {}
            for nombre in list(subidos):
                origen = Path(nombre)
                if origen.is_file():
                    os.replace(origen, DIR_ENTRADA / nombre_limpio(origen.stem + origen.suffix.lower()))
            del subidos
        elif pedir_archivos:
            print(f"(Fuera de Colab no hay botón de subida: copia tus archivos a {DIR_ENTRADA})")

    buenos, ignorados = listar_audios(carpeta)
    for p in ignorados:
        print(f"⚠️ Ignorado (no es wav/mp3/flac/m4a/ogg): {p.name}")
    ENTRADAS = [str(p) for p in buenos]
    if not ENTRADAS:
        print("❌ No hay canciones todavía. Sube archivos (o ponlos en la carpeta de Drive) y vuelve a ejecutar esta celda.")
        return
    print(f"\n✅ {len(ENTRADAS)} canción(es) en la lista:")
    for p in buenos:
        print(f"   • {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")
    print("Sigue con la celda 3.")


_entrada()

## 3 · ¿Qué quieres sacar?
Elige el objetivo y ejecuta. Procesa canción por canción; si una falla, sigue con las demás y lo dice en el resumen final. Si la vuelves a ejecutar, **se salta las canciones ya hechas** con las mismas opciones.

| Objetivo | Modelo que se elige (con GPU) | Por qué (SDR en MVSEP Multisong, más alto = mejor) |
|---|---|---|
| Voz + instrumental | MelBand Roformer de Kimberley Jensen | Voz 11.01 dB frente a 8.33 de `htdemucs_ft`. El instrumental es la mezcla menos la voz. |
| Batería / bajo / voz / otros | BS-Roformer-SW | Batería 14.11 vs 11.24, bajo 14.62 vs 12.05, otros 8.71 vs 5.74 frente a `htdemucs_ft`. |
| Además guitarra y piano | BS-Roformer-SW | Guitarra 9.05 y piano 7.83; `htdemucs_6s` queda atrás y su piano sale con artefactos. |
| Solo batería | BS-Roformer-SW | Mejor modelo público de batería (14.11 dB). |

- **Licencias:** BS-Roformer-SW es el de mejor calidad, pero sus pesos circulan **sin autor ni licencia declarados**. Con **«solo licencias claras»** se combina MelBand Roformer (saca la voz) con Demucs (separa el resto sobre el instrumental ya sin voz); ambos son MIT.
- **Sin GPU** se usa Demucs, porque los RoFormer en CPU tardan varias veces la duración de la canción.
- **Fuentes:** [tablas de MVSEP](https://mvsep.com/en/algorithms), [Demucs](https://github.com/adefossez/demucs), [audio-separator](https://github.com/nomadkaraoke/python-audio-separator), [procedencia de BS-Roformer-SW](https://github.com/ZFTurbo/Music-Source-Separation-Training/issues/248).

In [ ]:
#@title 3 · ¿Qué quieres sacar? (aquí se hace el trabajo) { display-mode: "form" }
#@markdown **Objetivo:** elige qué stems necesitas; el notebook escoge el mejor modelo para eso y te dice cuál usó y por qué.
objetivo = "Batería / bajo / voz / otros"  #@param ["Voz + instrumental", "Batería / bajo / voz / otros", "Además guitarra y piano", "Solo batería"]
#@markdown **Solo licencias claras:** evita BS-Roformer-SW (el de mejor calidad, pero con pesos de licencia desconocida) y usa MelBand Roformer + Demucs (MIT).
solo_licencias_claras = False  #@param {type:"boolean"}
#@markdown **Formato:** WAV de 24 bits (recomendado) o float de 32 bits. Siempre se conserva la frecuencia de muestreo original.
formato = "WAV 24 bits"  #@param ["WAV 24 bits", "WAV float 32 bits"]
#@markdown **Rehacer** aunque la canción ya esté separada con estas mismas opciones.
rehacer = False  #@param {type:"boolean"}
#@markdown ---
#@markdown ### Avanzado (puedes dejarlo como está)
#@markdown **Forzar modelo:** salta la elección automática.
forzar_modelo = "Automático (recomendado)"  #@param ["Automático (recomendado)", "BS-Roformer-SW", "MelBand Roformer Kim (voz) + Demucs (resto)", "Solo Demucs (htdemucs_ft / htdemucs_6s)", "Solo Demucs rápido (htdemucs)"]
#@markdown **`overlap_roformer`** (modelos RoFormer): 0 = el valor propio del modelo. Más alto (4, 8) = uniones más finas y proporcionalmente más lento.
overlap_roformer = 0  #@param {type:"slider", min:0, max:8, step:1}
#@markdown **`segment`** (Demucs, segundos por trozo): déjalo en 7.8, el máximo. Bajarlo casi no ahorra memoria y solo hace más lento el proceso.
segment = 7.8  #@param {type:"slider", min:2, max:7.8, step:0.1}
#@markdown **`shifts`** (Demucs, pasadas que se promedian): 1 = normal; 2 a 4 = un poco más limpio y 2 a 4 veces más lento. No gasta más memoria de GPU.
shifts = 1  #@param {type:"slider", min:0, max:5, step:1}
#@markdown **`overlap`** (Demucs, traslape entre trozos): 0.25 va bien; más alto = uniones más suaves y más lento.
overlap = 0.25  #@param {type:"slider", min:0.1, max:0.5, step:0.05}

_CLAVES_OBJETIVO = {"Voz + instrumental": "voz", "Batería / bajo / voz / otros": "4",
                    "Además guitarra y piano": "6", "Solo batería": "drums"}


def _firma(ruta, cfg, plan):
    info = os.stat(ruta)
    return {"origen": str(ruta), "tam": info.st_size, "mtime": int(info.st_mtime), "objetivo": plan["objetivo"],
            "familia": plan["familia"], **cfg}


def _ya_hecha(carpeta, firma):
    try:
        info = json.loads((carpeta / "info.json").read_text(encoding="utf-8"))
    except Exception:
        return None
    if info.get("firma") != firma or not info.get("archivos"):
        return None
    if not all((carpeta / a).is_file() for a in info["archivos"]):
        return None
    return info


def _precargar(plan, cfg):
    """Descarga/carga los modelos del plan antes de empezar, para fallar pronto y con un mensaje claro."""
    for clave in plan["modelos"]:
        if clave in ("sw", "kim"):
            cargar_roformer(clave, cfg["overlap_roformer"])
        else:
            cargar_demucs(clave)


def _procesar():
    global RESULTADOS, PLAN
    if "separar_cancion" not in globals():
        print("❌ Primero ejecuta la celda 1 (Preparación).")
        return
    if not globals().get("ENTRADAS"):
        print("❌ No hay canciones en la lista. Ejecuta la celda 2 (Entrada).")
        return

    cfg = {"formato": "float32" if "float" in formato else "pcm24", "segment": round(float(segment), 2),
           "shifts": int(shifts), "overlap": round(float(overlap), 2), "overlap_roformer": int(overlap_roformer)}
    hay_gpu = torch.cuda.is_available()
    dispositivo = "cuda" if hay_gpu else "cpu"
    forzar = {v: k for k, v in FAMILIAS.items()}.get(forzar_modelo)
    clave = _CLAVES_OBJETIVO[objetivo]
    plan = elegir_plan(clave, forzar, solo_licencias_claras, hay_gpu, HAY_ROFORMER)

    print(f"🎯 Objetivo: {objetivo}  →  stems: {', '.join(plan['salidas'])}  ·  {formato}")
    try:
        _precargar(plan, cfg)
    except BaseException as error:   # incluye fallos de descarga
        if isinstance(error, KeyboardInterrupt):
            raise
        motivo = explicar_error(error)
        if plan["usa_roformer"]:
            print(f"⚠️ No se pudo cargar {MODELOS[plan['modelos'][0]]['nombre']}: {motivo}\n   Se usa Demucs en su lugar.")
            descargar_modelos_de_memoria()
            plan = elegir_plan(clave, "demucs", False, hay_gpu, False)
            plan["motivo"] = "el modelo elegido no se pudo cargar, así que se usó Demucs como respaldo."
            try:
                _precargar(plan, cfg)
            except BaseException as error2:
                if isinstance(error2, KeyboardInterrupt):
                    raise
                print(f"❌ Tampoco se pudo cargar Demucs: {explicar_error(error2)}\n"
                      "   Suele ser un problema de conexión: vuelve a ejecutar esta celda.")
                return
        else:
            print(f"❌ No se pudo cargar el modelo: {motivo}\n"
                  "   Suele ser un problema de conexión al descargarlo: vuelve a ejecutar esta celda.")
            return
    PLAN = plan
    print(describir_plan(plan))
    if not hay_gpu:
        print("⚠️ Sin GPU: se procesa en CPU; calcula varios minutos por canción.")

    nombres = nombres_unicos(ENTRADAS)
    resultados, inicio_total = [], time.time()
    for n, ruta in enumerate(ENTRADAS, 1):
        cancion = nombres[ruta]
        carpeta = DIR_SALIDA / cancion
        res = {"cancion": cancion, "estado": "error", "detalle": "", "zip": None, "avisos": []}
        resultados.append(res)
        print(f"\n[{n}/{len(ENTRADAS)}] 🎵 {Path(ruta).name}")
        inicio = time.time()
        try:
            firma = _firma(ruta, cfg, plan)
            info = None if rehacer else _ya_hecha(carpeta, firma)
            if info is not None:
                res["estado"] = "ya estaba"
                print("   ⏭️ Ya estaba separada con estas opciones (marca «rehacer» para repetirla).")
            else:
                sin_memoria = False
                try:
                    datos = separar_cancion(ruta, cancion, carpeta, plan, cfg, dispositivo)
                except Exception as error:
                    if dispositivo == "cuda" and es_falta_de_memoria_gpu(error):
                        sin_memoria = True
                    else:
                        raise
                if sin_memoria:
                    descargar_modelos_de_memoria()
                    if plan["usa_roformer"]:
                        print("   ⚠️ Se acabó la memoria de la GPU: la libero y reintento esta canción…")
                        res["avisos"].append("hubo que liberar la memoria de la GPU y reintentar")
                        datos = separar_cancion(ruta, cancion, carpeta, plan, cfg, dispositivo)
                    else:
                        print("   ⚠️ Se acabó la memoria de la GPU con esta canción: la reintento en CPU (más lento)…")
                        res["avisos"].append("se procesó en CPU porque la GPU se quedó sin memoria")
                        datos = separar_cancion(ruta, cancion, carpeta, plan, cfg, "cpu")
                info = {"firma": firma, "modelos": [MODELOS[m]["nombre"] for m in plan["modelos"]], **datos}
                (carpeta / "info.json").write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding="utf-8")
                res["estado"] = "ok"
                print(f"   ✅ {len(datos['archivos'])} stem(s) · duración {fmt_tiempo(datos['duracion_s'])} · "
                      f"{datos['sr']} Hz · tardó {fmt_tiempo(time.time() - inicio)}")
                if datos["bloques"] > 1:
                    print(f"   ℹ️ Canción larga: se procesó en {datos['bloques']} bloques para cuidar la memoria.")
                if datos["ganancia_db"] < 0:
                    res["avisos"].append(f"se bajó {abs(datos['ganancia_db']):.1f} dB a todos sus stems para evitar distorsión")
                elif datos["pico"] > 1.0:
                    res["avisos"].append("tiene picos por encima de 0 dBFS (normal en float; baja el volumen en tu app)")
                if datos["sr"] != SR_TRABAJO:
                    res["avisos"].append(f"los modelos trabajan a {SR_TRABAJO} Hz; se convirtió y se devolvió a {datos['sr']} Hz")

            archivos = stems_de(carpeta, cancion)
            res["zip"], _ = crear_zip(DIR_ZIPS / f"{cancion}.zip", [(p, p.name) for p in archivos])
            res["carpeta"] = carpeta
            if globals().get("DRIVE_SALIDA"):
                try:
                    for p in archivos:
                        copiar_si_cambio(p, Path(DRIVE_SALIDA) / cancion / p.name)
                    print(f"   💾 Copiada a Drive: {Path(DRIVE_SALIDA) / cancion}")
                except Exception as error:
                    res["avisos"].append(f"no se pudo copiar a Drive ({explicar_error(error)})")
        except KeyboardInterrupt:
            print("\n⏹️ Detenido por ti. Lo ya terminado se conserva; vuelve a ejecutar la celda para continuar.")
            RESULTADOS = resultados
            return
        except Exception as error:
            res["estado"], res["detalle"] = "error", explicar_error(error)
            print(f"   ❌ Falló: {res['detalle']}\n   Sigo con las demás.")
            liberar_memoria()
        for aviso in res["avisos"]:
            print(f"   ⚠️ {aviso}")

    buenas = [r for r in resultados if r["estado"] != "error"]
    zip_todo = None
    if len(buenas) > 1:
        try:
            elementos = [(p, f"{r['cancion']}/{p.name}") for r in buenas for p in stems_de(r["carpeta"], r["cancion"])]
            zip_todo, _ = crear_zip(DIR_ZIPS / "todos_los_stems.zip", elementos)
        except Exception as error:
            print(f"⚠️ No se pudo crear el ZIP con todo: {explicar_error(error)}")
    RESULTADOS = resultados

    print("\n" + "=" * 60 + f"\nRESUMEN · tiempo total {fmt_tiempo(time.time() - inicio_total)}")
    print(describir_plan(plan) + "\n")
    iconos = {"ok": "✅", "ya estaba": "⏭️", "error": "❌"}
    for r in resultados:
        extra = f" — {r['detalle']}" if r["detalle"] else ""
        extra += "".join(f"\n      ⚠️ {a}" for a in r["avisos"])
        print(f" {iconos[r['estado']]} {r['cancion']} ({r['estado']}){extra}")
    fallas = len(resultados) - len(buenas)
    print(f"\n{len(buenas)} lista(s), {fallas} con error.")
    if buenas:
        print(f"ZIP por canción en: {DIR_ZIPS}" + (f"\nZIP con todo: {zip_todo}" if zip_todo else ""))
        print("Sigue con la celda 4 para escuchar y la 5 para descargar.")


_procesar()

## 4 · Escuchar
Reproductores para revisar cada stem. Por defecto suena un fragmento de 45 s (en MP3 ligero, solo para escuchar aquí); cambia el inicio o la duración si quieres oír otra parte.

In [ ]:
#@title 4 · Escuchar los stems antes de descargar { display-mode: "form" }
#@markdown Escribe parte del nombre para ver una sola canción (vacío = todas, máximo 6).
cancion = ""  #@param {type:"string"}
#@markdown Fragmento a escuchar: segundo de inicio y duración (duración 0 = canción completa, pesa más).
inicio_seg = 30  #@param {type:"integer"}
duracion_seg = 45  #@param {type:"integer"}
#@markdown Mostrar también la mezcla original para comparar.
incluir_original = True  #@param {type:"boolean"}


def _preview(origen, destino, inicio, duracion):
    """MP3 ligero solo para escuchar aquí (las descargas son los WAV completos)."""
    origen, destino = Path(origen), Path(destino)
    if destino.exists() and destino.stat().st_mtime >= origen.stat().st_mtime:
        return destino
    destino.parent.mkdir(parents=True, exist_ok=True)
    cmd = ["ffmpeg", "-nostdin", "-y", "-v", "error", "-ss", str(inicio)]
    if duracion > 0:
        cmd += ["-t", str(duracion)]
    cmd += ["-i", str(origen), "-vn", "-ac", "2", "-b:a", "192k", str(destino)]
    r = subprocess.run(cmd, capture_output=True)
    if r.returncode != 0 or not destino.exists() or destino.stat().st_size == 0:
        destino.unlink(missing_ok=True)
        return None
    return destino


def _escuchar():
    if "canciones_hechas" not in globals():
        print("❌ Primero ejecuta la celda 1 (Preparación).")
        return
    from IPython.display import Audio, HTML, display
    hechas = [c for c in canciones_hechas() if cancion.strip().lower() in c[0].lower()]
    if not hechas:
        print("❌ No hay stems para escuchar todavía" + (f" que coincidan con «{cancion}»." if cancion.strip() else ". Ejecuta la celda 3."))
        return
    if len(hechas) > 6:
        print(f"Hay {len(hechas)} canciones; muestro las primeras 6. Usa el campo «cancion» para ver otra.")
        hechas = hechas[:6]
    for nombre, carpeta, info in hechas:
        duracion_total = float(info.get("duracion_s", 0))
        inicio = max(0, int(inicio_seg))
        if duracion_total and inicio >= duracion_total - 5:
            inicio = 0
        dur = max(0, int(duracion_seg))
        tramo = f"desde {fmt_tiempo(inicio)}" + (f", {dur} s" if dur else ", completa")
        display(HTML(f"<h3 style='margin:16px 0 4px'>🎵 {nombre}</h3><small>{tramo}</small>"))
        pistas = [(p.stem[len(nombre) + 3:], p) for p in stems_de(carpeta, nombre)]
        origen = info.get("firma", {}).get("origen")
        if incluir_original and origen and Path(origen).is_file():
            pistas = [("mezcla original", Path(origen))] + pistas
        for etiqueta, ruta in pistas:
            mp3 = _preview(ruta, DIR_PREVIEWS / nombre / f"{etiqueta}__{inicio}_{dur}.mp3", inicio, dur)
            display(HTML(f"<b>{etiqueta}</b>"))
            if mp3 is None:
                print("   (no se pudo preparar la vista previa de este archivo)")
            else:
                display(Audio(filename=str(mp3), embed=True))


_escuchar()

## 5 · Descargar
Un clic: baja un ZIP con todo o un ZIP por canción, con los WAV en calidad completa. Los archivos se llaman `<canción> - drums.wav`, `<canción> - bass.wav`, `<canción> - vocals.wav`, `<canción> - other.wav` (más `- guitar.wav`, `- piano.wav` o `- instrumental.wav` según el objetivo).

In [ ]:
#@title 5 · Descargar { display-mode: "form" }
que_descargar = "Un ZIP con todo"  #@param ["Un ZIP con todo", "Un ZIP por canción"]


def _descargar():
    if "canciones_hechas" not in globals():
        print("❌ Primero ejecuta la celda 1 (Preparación).")
        return
    hechas = canciones_hechas()
    if not hechas:
        print("❌ Todavía no hay stems. Ejecuta la celda 3.")
        return
    try:
        zips = [crear_zip(DIR_ZIPS / f"{nombre}.zip", [(p, p.name) for p in stems_de(carpeta, nombre)])[0]
                for nombre, carpeta, _ in hechas]
        if que_descargar.startswith("Un ZIP con todo") and len(hechas) > 1:
            elementos = [(p, f"{nombre}/{p.name}") for nombre, carpeta, _ in hechas for p in stems_de(carpeta, nombre)]
            zips = [crear_zip(DIR_ZIPS / "todos_los_stems.zip", elementos)[0]]
    except Exception as error:
        print(f"❌ No se pudo preparar el ZIP: {explicar_error(error)}")
        return
    for z in zips:
        print(f"⬇️ {z.name}  ({z.stat().st_size / 1e6:.0f} MB)")
        if EN_COLAB:
            colab_files.download(str(z))
    if EN_COLAB:
        print("La descarga empieza sola; con archivos grandes Colab tarda un poco en prepararla. "
              "Si son varios ZIP, el navegador puede pedirte permiso para «descargar varios archivos».")
    else:
        print(f"(Fuera de Colab: los ZIP están en {DIR_ZIPS})")


_descargar()

## 6 · Limpieza (opcional)
Borra archivos temporales de la sesión para liberar espacio. Marca qué quieres borrar; **no toca tu Drive**.

In [ ]:
#@title 6 · Limpieza (opcional) { display-mode: "form" }
#@markdown Marca qué borrar **de la sesión de Colab**. Nunca se toca nada de tu Google Drive.
borrar_stems_y_zips = False  #@param {type:"boolean"}
borrar_canciones_subidas = False  #@param {type:"boolean"}
borrar_vistas_previas = True  #@param {type:"boolean"}
liberar_memoria_del_modelo = False  #@param {type:"boolean"}
#@markdown Los modelos RoFormer descargados ocupan ~0.7-0.9 GB cada uno; si los borras, se vuelven a descargar al usarlos.
borrar_modelos_descargados = False  #@param {type:"boolean"}


def _limpiar():
    if "DIR_SALIDA" not in globals():
        print("❌ Primero ejecuta la celda 1 (Preparación).")
        return
    global ENTRADAS

    def vaciar(carpeta):
        liberado = sum(p.stat().st_size for p in carpeta.rglob("*") if p.is_file())
        shutil.rmtree(carpeta, ignore_errors=True)
        carpeta.mkdir(parents=True, exist_ok=True)
        return liberado / 1e6

    if borrar_vistas_previas:
        print(f"🧹 Vistas previas borradas ({vaciar(DIR_PREVIEWS):.0f} MB).")
    if borrar_stems_y_zips:
        print(f"🧹 Stems y ZIPs borrados ({vaciar(DIR_SALIDA) + vaciar(DIR_ZIPS):.0f} MB).")
    if borrar_canciones_subidas:
        print(f"🧹 Canciones subidas borradas ({vaciar(DIR_ENTRADA):.0f} MB).")
        ENTRADAS = [r for r in globals().get("ENTRADAS", []) if Path(r).exists()]
    if liberar_memoria_del_modelo:
        descargar_modelos_de_memoria()
        print("🧹 Modelo descargado de la memoria (se vuelve a cargar solo cuando haga falta).")
    if borrar_modelos_descargados:
        descargar_modelos_de_memoria()
        print(f"🧹 Modelos descargados borrados ({vaciar(DIR_MODELOS):.0f} MB).")
    shutil.rmtree(DIR_TMP, ignore_errors=True)
    DIR_TMP.mkdir(parents=True, exist_ok=True)
    if not any([borrar_modelos_descargados, borrar_vistas_previas, borrar_stems_y_zips, borrar_canciones_subidas, liberar_memoria_del_modelo]):
        print("No marcaste nada: no se borró nada.")
    libre = shutil.disk_usage(BASE).free / 1e9
    print(f"Espacio libre en disco: {libre:.0f} GB.")


_limpiar()